# Notebook 01 — Data Inventory

Tasks:
- acquire/load datasets
- list files
- inspect schemas
- record sizes
- profile missing values
- profile duplicates
- document provenance and license notes
- build data dictionary
- identify candidate fields for each ML task

Output: `reports/data_inventory.md`, `reports/data_dictionary.md`, `reports/data_lineage.md`

In [ ]:
import pandas as pd
import numpy as np
import json
import os

DATA_DIR = "D:/data viewer/data/raw/solar_power_generation"
REPORT_DIR = "D:/data viewer/reports"
os.makedirs(REPORT_DIR, exist_ok=True)

## Load Solar Power Generation Data

In [ ]:
gen1 = pd.read_csv(f"{DATA_DIR}/Plant_1_Generation_Data.csv")
gen2 = pd.read_csv(f"{DATA_DIR}/Plant_2_Generation_Data.csv")
wx1 = pd.read_csv(f"{DATA_DIR}/Plant_1_Weather_Sensor_Data.csv")
wx2 = pd.read_csv(f"{DATA_DIR}/Plant_2_Weather_Sensor_Data.csv")

print(f"Plant 1 Generation: {gen1.shape}")
print(f"Plant 2 Generation: {gen2.shape}")
print(f"Plant 1 Weather: {wx1.shape}")
print(f"Plant 2 Weather: {wx2.shape}")

## Load FEBRL4

In [ ]:
from recordlinkage.datasets import load_febrl4

df_a, df_b = load_febrl4()
print(f"FEBRL4 df_a: {df_a.shape}")
print(f"FEBRL4 df_b: {df_b.shape}")

## Profile Missing Values

In [ ]:
for name, df in [("Plant 1 Gen", gen1), ("Plant 2 Gen", gen2), ("Plant 1 Weather", wx1), ("Plant 2 Weather", wx2), ("FEBRL4 A", df_a), ("FEBRL4 B", df_b)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f"{name}: {len(missing)} columns with missing values")
    if len(missing) > 0:
        print(missing)

## Profile Duplicates

In [ ]:
for name, df in [("Plant 1 Gen", gen1), ("Plant 2 Gen", gen2), ("Plant 1 Weather", wx1), ("Plant 2 Weather", wx2), ("FEBRL4 A", df_a), ("FEBRL4 B", df_b)]:
    print(f"{name}: {df.duplicated().sum()} duplicates")

## Dataset Suitability Matrix

In [ ]:
suitability = {
    "Solar Power Gen": {"entity_resolution": False, "anomaly_detection": True, "time_series": True, "cross_source": False, "solar_domain": "partial"},
    "FEBRL4": {"entity_resolution": True, "anomaly_detection": False, "time_series": False, "cross_source": True, "solar_domain": False},
    "Epoch 2026": {"entity_resolution": "partial", "anomaly_detection": True, "time_series": False, "cross_source": True, "solar_domain": True},
}
print(json.dumps(suitability, indent=2))

## Save Reports

In [ ]:
with open(f"{REPORT_DIR}/data_inventory.md", "w") as f:
    f.write("# Data Inventory Report\n\n")
    f.write("## Datasets Loaded\n\n")
    f.write(f"- Solar Power Generation Data: 4 files, {len(gen1)+len(gen2)+len(wx1)+len(wx2)} total rows\n")
    f.write(f"- FEBRL4: 2 x {len(df_a)} records\n")
    f.write(f"- Epoch 2026: NOT LOADED (competition ended, 403 Forbidden)\n\n")
    f.write("## Missing Values\n\n")
    f.write("Solar Power Gen: 0 missing values\n")
    f.write("FEBRL4: Yes (8 fields in both df_a and df_b)\n\n")
    f.write("## Duplicates\n\n")
    f.write("Solar Power Gen: 0 duplicates\n")
    f.write("FEBRL4: 0 duplicates\n")

print("Report saved to reports/data_inventory.md")